In [4]:
from pathlib import Path
import pandas as pd
import numpy as np

# 아래 경로는 이 노트북을 실행하는 작업 폴더입니다.
DATA_DIR = Path.cwd()
OUTPUT_DIR = DATA_DIR / "sampling_result"
OUTPUT_DIR.mkdir(exist_ok=True)

INPUT_FILES = {
    1: "pikmin_reviews1.csv",
    2: "pikmin_reviews2.csv",
    3: "pikmin_reviews3.csv",
    4: "pikmin_reviews4.csv",
    5: "pikmin_reviews5.csv",
}

# 이전 파일럿 100건: 파일 이름이 다른 경우 수정
PILOT_FILE = "pokemon_go_pilot_coding_100.csv"
EXCLUDE_PILOT = True  # 이전 파일럿 100건을 본 분석에서 제외
RANDOM_STATE = 42
N_MAIN_PER_RATING = 400
N_RELIABILITY_PER_RATING = 100

print("작업 폴더:", DATA_DIR)
print("출력 폴더:", OUTPUT_DIR)

작업 폴더: d:\vscode_project\포켓몬고
출력 폴더: d:\vscode_project\포켓몬고\sampling_result


In [5]:
import re

def read_csv_robust(path):
    for encoding in ("utf-8-sig", "utf-8", "cp949"):
        try:
            return pd.read_csv(path, encoding=encoding, dtype=str, keep_default_na=False)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"파일 인코딩을 확인해 주세요: {path}")

frames = []
source_log = []
for rating, filename in INPUT_FILES.items():
    file_path = DATA_DIR / filename
    if not file_path.is_file():
        raise FileNotFoundError(f"원본 파일을 찾지 못했습니다: {file_path}")
    data = read_csv_robust(file_path)
    required_columns = {"ID", "날짜", "내용"}
    missing = required_columns - set(data.columns)
    if missing:
        raise ValueError(f"{filename} 필수 열 없음: {missing}")
    if "별점" in data.columns:
        parsed = data["별점"].str.extract(r"만점에\s*([1-5])개")[0]
        bad = parsed.notna() & (parsed.astype("string") != str(rating))
        if bad.any():
            raise ValueError(f"{filename}: 파일 별점과 다른 리뷰 {int(bad.sum())}건")
    if (data["ID"].str.strip() == "").any():
        raise ValueError(f"{filename}: 리뷰 ID가 비어 있는 행이 있습니다.")
    if (data["내용"].str.strip() == "").any():
        raise ValueError(f"{filename}: 리뷰 본문이 비어 있는 행이 있습니다.")
    data["별점_숫자"] = rating
    data["원본파일"] = filename
    data["원본행"] = np.arange(2, len(data) + 2)
    frames.append(data)
    source_log.append({"별점": rating, "원본건수": len(data), "입력파일": filename})

all_reviews = pd.concat(frames, ignore_index=True)
if all_reviews["ID"].duplicated().any():
    repeated = all_reviews[all_reviews["ID"].duplicated(keep=False)]
    raise ValueError(f"전체 원자료에 중복 ID {len(repeated)}행이 있습니다. 중복을 확인한 뒤 진행하세요.")

print("전체 원본 리뷰:", len(all_reviews), "건")
print(pd.DataFrame(source_log).to_string(index=False))
print("본문 중복(참고):", int(all_reviews["내용"].duplicated().sum()), "건")


전체 원본 리뷰: 15040 건
 별점  원본건수                입력파일
  1  2800 pikmin_reviews1.csv
  2  3160 pikmin_reviews2.csv
  3  3120 pikmin_reviews3.csv
  4  3080 pikmin_reviews4.csv
  5  2880 pikmin_reviews5.csv
본문 중복(참고): 3 건


In [7]:
pilot_ids = set()
if EXCLUDE_PILOT:
    pilot_path = DATA_DIR / PILOT_FILE
    if not pilot_path.is_file():
        raise FileNotFoundError(f"파일럿 제외를 위해 다음 파일이 필요합니다: {pilot_path}")
    pilot = read_csv_robust(pilot_path)
    if "ID" not in pilot.columns:
        raise ValueError("파일럿 CSV에는 ID 열이 있어야 합니다.")
    pilot_ids = set(pilot["ID"].str.strip()) - {""}
    print("파일럿 고유 ID:", len(pilot_ids), "건")

available = all_reviews.loc[~all_reviews["ID"].isin(pilot_ids)].copy()
excluded = all_reviews.loc[all_reviews["ID"].isin(pilot_ids)].copy()
print("실제 제외된 파일럿 리뷰:", len(excluded), "건")
print("추출 가능 리뷰:", len(available), "건")

minimum = N_MAIN_PER_RATING + N_RELIABILITY_PER_RATING
for rating in range(1, 6):
    count = int((available["별점_숫자"] == rating).sum())
    if count < minimum:
        raise ValueError(f"{rating}점: {count}건만 남아 {minimum}건을 추출할 수 없습니다.")


파일럿 고유 ID: 100 건
실제 제외된 파일럿 리뷰: 100 건
추출 가능 리뷰: 14940 건


In [8]:
main_parts, reliability_parts = [], []

for rating in range(1, 6):
    group = available.loc[available["별점_숫자"] == rating]
    # 동일한 파일 및 동일한 난수 시드를 사용하면 재현되는 비복원 표본
    selected = group.sample(n=minimum, replace=False, random_state=RANDOM_STATE + rating)
    main_parts.append(selected.iloc[:N_MAIN_PER_RATING].copy())
    reliability_parts.append(selected.iloc[N_MAIN_PER_RATING:].copy())

main_2000 = pd.concat(main_parts, ignore_index=True)
reliability_500 = pd.concat(reliability_parts, ignore_index=True)

# 무작위로 뽑은 순서를 그대로 보존하되, 추적을 위한 표본 ID 부여
main_2000.insert(0, "표본번호", [f"M{i:04d}" for i in range(1, len(main_2000) + 1)])
reliability_500.insert(0, "표본번호", [f"R{i:04d}" for i in range(1, len(reliability_500) + 1)])

# 코딩을 위한 판정 칸은 빈 값으로 둡니다. 코드 체계는 코딩북 V1.1 기준입니다.
coding_columns = [f"A{i:02d}" for i in range(1, 10)] + ["G", "B01", "코더메모"]
for df_part in (main_2000, reliability_500):
    for column in coding_columns:
        df_part[column] = ""

print("본 코딩:", len(main_2000), "건")
print("신뢰도:", len(reliability_500), "건")


본 코딩: 2000 건
신뢰도: 500 건


In [9]:
assert len(main_2000) == 2000
assert len(reliability_500) == 500
assert main_2000["ID"].is_unique
assert reliability_500["ID"].is_unique
assert set(main_2000["ID"]).isdisjoint(set(reliability_500["ID"]))
assert set(main_2000["ID"]).isdisjoint(pilot_ids)
assert set(reliability_500["ID"]).isdisjoint(pilot_ids)

for rating in range(1, 6):
    assert (main_2000["별점_숫자"] == rating).sum() == 400
    assert (reliability_500["별점_숫자"] == rating).sum() == 100

summary = pd.DataFrame({
    "별점": list(range(1, 6)),
    "원본건수": [int((all_reviews["별점_숫자"] == r).sum()) for r in range(1, 6)],
    "파일럿제외": [int((excluded["별점_숫자"] == r).sum()) for r in range(1, 6)],
    "코딩2000": [int((main_2000["별점_숫자"] == r).sum()) for r in range(1, 6)],
    "신뢰도500": [int((reliability_500["별점_숫자"] == r).sum()) for r in range(1, 6)],
})
print(summary.to_string(index=False))
print("✅ 모든 표본 수, 중복, 파일럿 제외 조건 통과")


 별점  원본건수  파일럿제외  코딩2000  신뢰도500
  1  2800     20     400     100
  2  3160     20     400     100
  3  3120     20     400     100
  4  3080     20     400     100
  5  2880     20     400     100
✅ 모든 표본 수, 중복, 파일럿 제외 조건 통과


In [10]:
# 원본 리뷰 식별자와 코딩 열을 앞에 배치합니다.
front = ["표본번호", "별점_숫자", "ID", "날짜", "내용"]
for dataset in (main_2000, reliability_500):
    other = [c for c in dataset.columns if c not in front + coding_columns]
    dataset[:]  # 데이터는 변경하지 않습니다.

main_cols = front + coding_columns + [c for c in main_2000.columns if c not in front + coding_columns]
reliability_cols = front + coding_columns + [c for c in reliability_500.columns if c not in front + coding_columns]

main_path = OUTPUT_DIR / "PokemonGO_Main_2000.csv"
reliability_path = OUTPUT_DIR / "PokemonGO_Reliability_500.csv"
log_path = OUTPUT_DIR / "PokemonGO_Sampling_Log.csv"

main_2000[main_cols].to_csv(main_path, index=False, encoding="utf-8-sig")
reliability_500[reliability_cols].to_csv(reliability_path, index=False, encoding="utf-8-sig")
summary.to_csv(log_path, index=False, encoding="utf-8-sig")

print("저장 완료:")
print(main_path)
print(reliability_path)
print(log_path)
print("\n미리 보기:")
print(main_2000[["표본번호", "별점_숫자", "ID", "내용"]].head(3).to_string(index=False, max_colwidth=35))


저장 완료:
d:\vscode_project\포켓몬고\sampling_result\PokemonGO_Main_2000.csv
d:\vscode_project\포켓몬고\sampling_result\PokemonGO_Reliability_500.csv
d:\vscode_project\포켓몬고\sampling_result\PokemonGO_Sampling_Log.csv

미리 보기:
 표본번호  별점_숫자                                  ID                                  내용
M0001      1 bf6d4126-5082-4adb-aa5e-83f56e42... 포켓몬 검색하면 자꾸 포켓몬 목록이 엄청 작게나오는 버그가...
M0002      1 12eab89b-4930-4bf6-afe4-9c50c771... 포켓몬홈과 연동을 하려하는데 왜 연동도 안되고 error창...
M0003      1 1909b814-2134-4d40-8fe2-072065ee... 하 들어가는데 안 들어가져서 로그아웃 눌렀다가 다시 들어갈...
